# Mission 1: The Keyword Detective 🔎
**Build a school FAQ search engine.** Suggested schedule: two 60-minute Friday sessions. 

**You will learn**: (1) how a computer stores questions and answers; (2) why keyword search can be fooled; (3) what *TF-IDF* and *cosine similarity* mean; (4) why an assistant needs permission to say “I don't know.”

**Rules:** All school rules below are **fictional**. They are for learning, not real school guidance. Do not enter names, passwords, or private school documents.

**Before running code:** Predict: Will “Where can I check out a novel?” match a school FAQ about “borrow a book”? Why?

## 1. Setup (teacher runs once)
 We use **scikit-learn**, a Python library with ready-made machine-learning tools. In Google Colab it is usually preinstalled. If the next cell gives `ModuleNotFoundError`, ask your teacher to run `%pip install scikit-learn`.

In [1]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print("Ready to investigate!")

Ready to investigate!


## 2. Give the computer a tiny knowledge base
 A **knowledge base** is a collection of information the assistant can search. These are **made-up** school examples.

 **Your turn:** Change an answer or add an entirely new FAQ. Notice: the computer can search only what you give it.

In [2]:
faqs = [
    {"topic": "library", "question": "How can I borrow a book?", "answer": "Visit the library desk with your school ID."},
    {"topic": "library hours", "question": "When does the library close?", "answer": "The fictional library closes at 4 PM on school days."},
    {"topic": "lost and found", "question": "Where is the lost and found?", "answer": "Go to the fictional main office, Room 101."},
    {"topic": "homework help", "question": "When can I get homework help?", "answer": "The fictional homework club meets Tuesdays in Room 12."},
    {"topic": "lunch", "question": "When is lunch?", "answer": "At our imaginary school, lunch begins at 12:15 PM."},
    {"topic": "AI policy", "question": "Can I use AI to complete homework?", "answer": "Ask your teacher. Use AI only when your teacher permits it and credit the help you receive."},
]
print("Number of FAQs:", len(faqs))
print("First FAQ:", faqs[0]["question"])

Number of FAQs: 6
First FAQ: How can I borrow a book?


## 3. First build a *very simple* keyword search
 This version counts words shared by the question and FAQ. It ignores punctuation and common words.

 **Prediction:** What happens when you ask “Where is my missing jacket?” instead of “Where is the lost and found?”

In [3]:
def words(text):
    common = {"the", "a", "an", "is", "in", "on", "at", "i", "my", "can", "how", "when", "where", "do", "does", "what", "to"}
    return set(re.findall(r"[a-z]+", text.lower())) - common

def simple_search(question):
    question_words = words(question)
    results = []
    for faq in faqs:
        shared = question_words & words(faq["question"])
        results.append((len(shared), faq["question"], faq["answer"]))
    return sorted(results, reverse=True)

for score, question, answer in simple_search("Where is the lost and found?")[:2]:
    print(score, "shared words |", question)

print("Try a different phrase:")
for score, question, answer in simple_search("Where is my missing jacket?")[:2]:
    print(score, "shared words |", question)

3 shared words | Where is the lost and found?
0 shared words | When is lunch?
Try a different phrase:
0 shared words | Where is the lost and found?
0 shared words | When is lunch?


### Think like a scientist
 **Observation:** Two questions can mean the same thing but use different words. That's called **vocabulary mismatch**.

 Now try these queries in the next cell: `"borrow a book"`, `"check out a novel"`, `"library closes"`, and `"missing backpack"`.

In [4]:
my_question = "borrow a book"  # CHANGE ME
for score, question, answer in simple_search(my_question)[:3]:
    print(f"Shared words: {score} | FAQ: {question}")

Shared words: 2 | FAQ: How can I borrow a book?
Shared words: 0 | FAQ: Where is the lost and found?
Shared words: 0 | FAQ: When is lunch?


## 4. Level up: TF-IDF + cosine similarity
 **TF-IDF** means “term frequency–inverse document frequency.” In simple terms, **rare, important words count more than very common words**.

 **Cosine similarity** compares two lists of numbers and gives us a score for how closely they point in the same direction. Here, we use it to compare the words in questions. The score is *not* a probability that an answer is correct.

In [5]:
# Convert each FAQ question into numbers based on its words.
vectorizer = TfidfVectorizer(stop_words="english")
faq_vectors = vectorizer.fit_transform([item["question"] for item in faqs])

def keyword_search(question):
    question_vector = vectorizer.transform([question])
    scores = cosine_similarity(question_vector, faq_vectors)[0]
    best_index = int(np.argmax(scores))
    return {"score": float(scores[best_index]), **faqs[best_index]}

for question in ["borrow a book", "check out a novel", "library closes", "missing backpack"]:
    result = keyword_search(question)
    print(f"{question!r} -> {result['topic']!r} (similarity {result['score']:.2f})")

'borrow a book' -> 'library' (similarity 1.00)
'check out a novel' -> 'library' (similarity 0.00)
'library closes' -> 'library hours' (similarity 0.58)
'missing backpack' -> 'library' (similarity 0.00)


## 5. The “I don't know” button
 A responsible assistant should not invent school rules. Our **threshold** is a minimum similarity score. It is a simple warning signal, *not proof* of correctness.

 **Your turn:** Change `MIN_SCORE` to `0.0`, then `0.30`. Which value is more cautious?

In [6]:
MIN_SCORE = 0.15  # CHANGE ME: try 0.0 and 0.30
def answer_with_keyword_search(question):
    match = keyword_search(question)
    if match["score"] < MIN_SCORE:
        return "I don't know based on these FAQs. Ask a school staff member."
    return match["answer"] + f"  [FAQ: {match['topic']}]"

for question in ["When is lunch?", "Where can I find my lost jacket?", "Does this school have a swimming pool?"]:
    print("Q:", question)
    print("A:", answer_with_keyword_search(question), "\n")

Q: When is lunch?
A: At our imaginary school, lunch begins at 12:15 PM.  [FAQ: lunch] 

Q: Where can I find my lost jacket?
A: Go to the fictional main office, Room 101.  [FAQ: lost and found] 

Q: Does this school have a swimming pool?
A: The fictional library closes at 4 PM on school days.  [FAQ: library hours] 



### Spot the mistake!
If the swimming-pool question produced a library answer, **that is a false match, not a school rule**. A generic overlapping word can trick keyword retrieval, and a nonzero score does **not** prove relevance.

**Challenge:** Raise `MIN_SCORE` and rerun. Is the bad answer rejected? What legitimate questions would a stricter threshold accidentally reject? This trade-off is why humans should verify answers and why we'll study meaning in Mission 2.

## 6. Check your work
 These tests check the **mechanics**, not whether every answer is factually correct.

In [7]:
assert len(faqs) >= 6, "Keep the six starting FAQs."
assert keyword_search("When is lunch?")["topic"] == "lunch"
assert keyword_search("borrowing books")["score"] >= 0.0
assert isinstance(answer_with_keyword_search("mystery question"), str)
print("✅ Mission 1 checks passed!")

✅ Mission 1 checks passed!


## Exit ticket (no code)
 1. Why can a keyword system miss two questions that mean the same thing?
 2. What is TF-IDF trying to emphasize?
 3. Why does a confidence threshold **not** guarantee a correct answer?
 4. Name one case where the assistant should say “I don't know.”

 **Next:** In Mission 2, we give the computer a way to compare *meaning*, not just identical words.